### Forecasting Notebook – Clean Version

#### Cell 1 – Imports & Configuration

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split  

from xgboost import XGBRegressor
from prophet import Prophet
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# paths
DATA_PATH = '../data/world_trade_data_features.csv'
CLASSIF_PATH = '../data/res/classification_results.csv'
OUT_DIR = '../data/res'
EVAL_DIR = os.path.join(OUT_DIR, 'evaluation')
DASH_DIR = os.path.join(OUT_DIR, 'dashboard')
MAPPINGS_PATH = os.path.join(OUT_DIR, 'label_mappings.json')

TRAIN_START, TRAIN_END = 2012, 2021
VAL_YEAR = 2022
TEST_START, TEST_END = 2023, 2024
FORECAST_YEARS = [2025, 2026, 2027]
COVID_YEARS = [2020, 2021]

SEQ_LEN = 3 
PROPHET_UNDERFIT_MAPE_THRESHOLD = 50.0  

os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(DASH_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


#### Cell 2 – Load Data & Basic Cleaning

In [4]:
df = pd.read_csv(DATA_PATH)

df = df.sort_values(['importer', 'product', 'year']).reset_index(drop=True)

df['is_covid_year'] = df['year'].isin(COVID_YEARS)

growth_cols = ['world_demand_growth', 'world_demand_growth_3y', 'algeria_export_growth', 'global_demand_growth']
for col in growth_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

eco_cols = ['gdp_usd', 'gdp_per_capita', 'gdp_growth_rate', 'population', 'inflation_rate', 'trade_openness']
for col in eco_cols:
    if col in df.columns:
        df[col] = df.groupby('importer')[col].ffill().bfill()

df = df.fillna(df.median(numeric_only=True))

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Data loaded: 1501178 rows, 42 columns


#### Cell 3 – Feature Engineering (Lags & Rolling Means)

In [5]:
def add_lag_features(data, lags=[1,2,3]):
    """Add lagged total_value, world_demand_growth, and global_demand_v."""
    data = data.copy()
    for lag in lags:
        data[f'total_value_lag_{lag}'] = data.groupby(['importer', 'product'])['total_value'].shift(lag)
        if 'world_demand_growth' in data.columns:
            data[f'world_demand_growth_lag_{lag}'] = data.groupby(['importer', 'product'])['world_demand_growth'].shift(lag)
        if 'global_demand_v' in data.columns:
            data[f'demand_lag_{lag}'] = data.groupby(['importer', 'product'])['global_demand_v'].shift(lag)
    # Rolling mean
    data['world_demand_rolling_3y'] = data.groupby(['importer', 'product'])['total_value'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
    # Fill all lag NaNs with 0
    lag_cols = [f'total_value_lag_{lag}' for lag in lags] + \
               ([f'world_demand_growth_lag_{lag}' for lag in lags] if 'world_demand_growth' in data.columns else []) + \
               ([f'demand_lag_{lag}' for lag in lags] if 'global_demand_v' in data.columns else []) + \
               ['world_demand_rolling_3y']
    for col in lag_cols:
        if col in data.columns:
            data[col] = data[col].fillna(0)
    return data

df = add_lag_features(df)
print("Lag features added.")

Lag features added.


#### Cell 4 – Encode Categorical Variables (Save Mappings)

In [6]:
categorical_cols = ['continent', 'official_language', 'main_spoken_language', 'sector']
all_mappings = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    all_mappings[col] = {
        "code_to_label": {int(i): label for i, label in enumerate(le.classes_)},
        "label_to_code": {label: int(i) for i, label in enumerate(le.classes_)}
    }

with open(MAPPINGS_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_mappings, f, ensure_ascii=False, indent=2)

print(f"Saved label mappings to {MAPPINGS_PATH}")

Saved label mappings to ../data/res/label_mappings.json


#### Cell 5 – Define Time Split

In [7]:
def time_split(df):
    train = df[(df['year'] >= TRAIN_START) & (df['year'] <= TRAIN_END)]
    val   = df[df['year'] == VAL_YEAR]
    test  = df[(df['year'] >= TEST_START) & (df['year'] <= TEST_END)]
    return train, val, test

# Example (will be reused later)
print("Time split defined: train 2012-2021, val 2022, test 2023-2024")

Time split defined: train 2012-2021, val 2022, test 2023-2024


#### Cell 6 – Filter Helpers (Task 1 & Task 2)

In [8]:
def filter_sparse_series(df, target_col, min_nonzero=5):
    """Keep only (importer, product) pairs with at least min_nonzero non-zero target values."""
    non_zero = df[df[target_col] != 0]
    counts = non_zero.groupby(['importer', 'product']).size().reset_index(name='n')
    good_pairs = counts[counts['n'] >= min_nonzero][['importer', 'product']]
    return df.merge(good_pairs, on=['importer', 'product'], how='inner')

def filter_top_algeria_series(df, top_n=50):
    """For Task 2: keep only rows where algeria_export_v > 0, then keep top_n pairs by cumulative export value."""
    algeria_only = df[df['algeria_export_v'] > 0].copy()
    pair_totals = algeria_only.groupby(['importer', 'product'])['algeria_export_v'].sum().reset_index()
    top_pairs = pair_totals.sort_values('algeria_export_v', ascending=False).head(top_n)[['importer', 'product']]
    return algeria_only.merge(top_pairs, on=['importer', 'product'], how='inner')

#### Cell 7 – Prophet Runner (Generic)

In [ ]:
def run_prophet_series(series_df, target_col, forecast_years, use_covid=True):
    """
    Fit Prophet on a single (importer, product) series.
    Returns forecast DataFrame for all years (including history).
    """
    pdf = series_df[['year', target_col]].copy()
    pdf['ds'] = pd.to_datetime(pdf['year'], format='%Y')
    pdf['y']  = pdf[target_col].clip(lower=0)
    pdf = pdf.dropna(subset=['y'])

    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.3,
        interval_width=0.80
    )
    if use_covid and 'is_covid_year' in series_df.columns:
        model.add_regressor('is_covid_year')
        pdf['is_covid_year'] = series_df.set_index('year')['is_covid_year'].reindex(pdf['year']).fillna(0).astype(int)

    model.fit(pdf[['ds', 'y'] + (['is_covid_year'] if use_covid and 'is_covid_year' in series_df.columns else [])])

    future_years = sorted(set(pdf['year'].tolist()) | set(forecast_years))
    future = pd.DataFrame({'ds': pd.date_range(start=f'{min(future_years)}-01-01', 
                                               end=f'{max(future_years)}-01-01', freq='YS')})
    future = future[future['ds'].dt.year.isin(future_years)]
    if use_covid and 'is_covid_year' in series_df.columns:
        future['is_covid_year'] = future['ds'].dt.year.isin(COVID_YEARS).astype(int)

    forecast = model.predict(future)
    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

#### Cell 8 – LSTM Runner (Generic)

In [10]:
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def prepare_lstm_sequences(arr, target_idx=0, seq_len=SEQ_LEN):
    X, y = [], []
    for i in range(len(arr) - seq_len):
        X.append(arr[i:i+seq_len, :])
        y.append(arr[i+seq_len, target_idx])
    return np.array(X), np.array(y)

def run_lstm_series(series_df, target_col, feature_cols, forecast_years, seq_len=SEQ_LEN):
    """
    Train LSTM on a single series (training only on years <= TRAIN_END),
    then predict forecast_years iteratively.
    Returns dict with forecasts and scaler.
    """
    # Prepare data
    series = series_df.sort_values('year')
    years = series['year'].values
    
    # Use only years up to TRAIN_END for training (no leakage)
    train_mask = years <= TRAIN_END
    train_df = series[train_mask].copy()
    
    if len(train_df) < seq_len + 1:
        return None
    
    # Scale features
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(train_df[feature_cols].values)
    
    X, y = prepare_lstm_sequences(scaled, target_idx=0, seq_len=seq_len)
    if len(X) == 0:
        return None
    
    model = build_lstm_model(input_shape=(seq_len, len(feature_cols)))
    es = EarlyStopping(patience=10, restore_best_weights=True, verbose=0)
    model.fit(X, y, epochs=100, batch_size=4, callbacks=[es], verbose=0)
    
    # Forecast future
    # Use last seq_len rows of the full series (including val/test if available) to start
    last_known = series[feature_cols].values[-seq_len:]
    last_scaled = scaler.transform(last_known)
    
    future_preds = []
    for _ in forecast_years:
        pred_scaled = model.predict(last_scaled[np.newaxis], verbose=0)[0, 0]
        future_preds.append(pred_scaled)
        # Update sequence: drop first, append new row (with new target and copy other features)
        new_row = last_scaled[-1].copy()
        new_row[0] = pred_scaled
        last_scaled = np.vstack([last_scaled[1:], new_row])
    
    # Inverse transform
    def inv_target(arr_scaled):
        dummy = np.zeros((len(arr_scaled), len(feature_cols)))
        dummy[:, 0] = arr_scaled
        return scaler.inverse_transform(dummy)[:, 0]
    
    forecast_values = inv_target(np.array(future_preds))
    return dict(zip(forecast_years, forecast_values))

#### Cell 9 – Evaluation Helper

In [11]:
def evaluate_forecast(y_true, y_pred, label=''):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan
    if label:
        print(f"{label:20s} MAE={mae:12,.0f} RMSE={rmse:12,.0f} MAPE={mape:6.1f}%")
        return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
    return mae, rmse, mape

#### Cell 10 – Task 1 Data Preparation

In [12]:
# For Task 1: world demand forecast (total_value)
df_t1 = filter_sparse_series(df, target_col='total_value', min_nonzero=5)
train_t1, val_t1, test_t1 = time_split(df_t1)

print(f"Task1 rows: total {len(df_t1)} | train {len(train_t1)} | val {len(val_t1)} | test {len(test_t1)}")
print(f"Number of (importer, product) pairs: {df_t1.groupby(['importer','product']).ngroups}")

Task1 rows: total 1497312 | train 1152090 | val 115279 | test 229943
Number of (importer, product) pairs: 117043


#### Cell 11 – Task 1: Prophet on All Series + Identify Underfitters

In [ ]:
def compute_prophet_metrics(df_full, target_col, val_split_df, test_split_df, forecast_years):
    """
    Run Prophet on every (importer, product) pair.
    Returns:
        forecasts: dict {(imp,prod): forecast_df (for all years)}
        metrics: dict {'val': list of dicts, 'test': list of dicts}
        underfit_pairs: set of pairs where val MAPE > threshold
    """
    pairs = df_full.groupby(['importer', 'product']).size().index.tolist()
    forecasts = {}
    metrics = {'val': [], 'test': []}
    
    for imp, prod in tqdm(pairs, desc="Prophet"):
        series_all = df_full[(df_full['importer']==imp) & (df_full['product']==prod)].sort_values('year')
        if len(series_all) < 5:
            continue
        try:
            fc_df = run_prophet_series(series_all, target_col, forecast_years=forecast_years)
            fc_df['year'] = fc_df['ds'].dt.year
            forecasts[(imp, prod)] = fc_df
            
            # Validation (2022)
            actual_val = val_split_df[(val_split_df['importer']==imp) & (val_split_df['product']==prod)]
            if not actual_val.empty:
                pred_val = fc_df[fc_df['year'] == VAL_YEAR]
                if not pred_val.empty:
                    y_true = actual_val[target_col].values
                    y_pred = pred_val['yhat'].values
                    mae, rmse, mape = evaluate_forecast(y_true, y_pred, label='')
                    metrics['val'].append({'importer':imp, 'product':prod, 'MAE':mae, 'RMSE':rmse, 'MAPE':mape})
            
            # Test (2023-2024)
            actual_test = test_split_df[(test_split_df['importer']==imp) & (test_split_df['product']==prod)]
            if not actual_test.empty:
                pred_test = fc_df[fc_df['year'].isin([2023,2024])]
                if not pred_test.empty:
                    y_true = actual_test[target_col].values
                    y_pred = pred_test['yhat'].values
                    mae, rmse, mape = evaluate_forecast(y_true, y_pred, label='')
                    metrics['test'].append({'importer':imp, 'product':prod, 'MAE':mae, 'RMSE':rmse, 'MAPE':mape})
        except Exception as e:
            continue
    
    # Identify underfitters (val MAPE > threshold)
    underfit = set()
    for rec in metrics['val']:
        if not np.isnan(rec['MAPE']) and rec['MAPE'] > PROPHET_UNDERFIT_MAPE_THRESHOLD:
            underfit.add((rec['importer'], rec['product']))
    
    return forecasts, metrics, underfit

print("Running Prophet on Task 1 (world demand)...")
prophet_fc_t1, prophet_metrics_t1, underfit_t1 = compute_prophet_metrics(
    df_t1, 'total_value', val_t1, test_t1, forecast_years=list(range(TRAIN_START, FORECAST_YEARS[-1]+1))
)

print(f"Prophet Task1 – {len(prophet_fc_t1)} pairs forecasted. Underfitters: {len(underfit_t1)}")

#### Cell 12 – Task 1: LSTM on Underfitting Series

In [ ]:
lstm_features_t1 = ['total_value', 'world_demand_growth', 'gdp_growth_rate', 
                    'inflation_rate', 'trade_openness', 'is_covid_year']
lstm_features_t1 = [c for c in lstm_features_t1 if c in df_t1.columns]

lstm_fc_t1 = {}  # only for underfitters
if underfit_t1:
    print(f"Running LSTM on {len(underfit_t1)} underfitting pairs for Task1...")
    for imp, prod in tqdm(underfit_t1):
        series = df_t1[(df_t1['importer']==imp) & (df_t1['product']==prod)].sort_values('year')
        if len(series) < SEQ_LEN + 3:
            continue
        fc = run_lstm_series(series, 'total_value', lstm_features_t1, FORECAST_YEARS)
        if fc:
            lstm_fc_t1[(imp, prod)] = fc
else:
    print("No underfitting pairs for Task1 (Prophet sufficient).")

#### Cell 13 – Task 1: Combine Final Forecasts for Dashboard

In [ ]:
# Use Prophet for all, override with LSTM for underfitters
final_fc_t1 = {}
for (imp, prod), fc_df in prophet_fc_t1.items():
    if (imp, prod) in underfit_t1 and (imp, prod) in lstm_fc_t1:
        # Replace 2025-2027 values with LSTM forecast
        for yr, val in lstm_fc_t1[(imp, prod)].items():
            fc_df.loc[fc_df['year'] == yr, 'yhat'] = val
            # Also set lower/upper to same (no intervals)
            fc_df.loc[fc_df['year'] == yr, 'yhat_lower'] = val * 0.8
            fc_df.loc[fc_df['year'] == yr, 'yhat_upper'] = val * 1.2
    final_fc_t1[(imp, prod)] = fc_df

print(f"Task1 final forecasts prepared for {len(final_fc_t1)} pairs.")

#### Cell 14 – Task 1 Evaluation (Aggregate & Per-Pair)

In [ ]:
print("\n=== TASK 1 EVALUATION (World Demand) ===")

# Validation metrics from Prophet (all pairs)
val_df_t1 = pd.DataFrame(prophet_metrics_t1['val'])
if not val_df_t1.empty:
    print("\nProphet – Validation 2022 (all pairs):")
    print(f"  Avg MAE : {val_df_t1['MAE'].mean():14,.0f}")
    print(f"  Avg RMSE: {val_df_t1['RMSE'].mean():14,.0f}")
    print(f"  Avg MAPE: {val_df_t1['MAPE'].mean():10.1f}%")

# Test metrics from Prophet (all pairs)
test_df_t1 = pd.DataFrame(prophet_metrics_t1['test'])
if not test_df_t1.empty:
    print("\nProphet – Test 2023-2024 (all pairs):")
    print(f"  Avg MAE : {test_df_t1['MAE'].mean():14,.0f}")
    print(f"  Avg RMSE: {test_df_t1['RMSE'].mean():14,.0f}")
    print(f"  Avg MAPE: {test_df_t1['MAPE'].mean():10.1f}%")

# LSTM on underfitters only (no separate aggregate needed)

#### Cell 15 – Task 2 Data Preparation (Top 50 Algeria series)

In [ ]:
df_t2 = filter_top_algeria_series(df, top_n=50)
train_t2, val_t2, test_t2 = time_split(df_t2)

print(f"Task2 rows: total {len(df_t2)} | train {len(train_t2)} | val {len(val_t2)} | test {len(test_t2)}")
print(f"Number of (importer, product) pairs: {df_t2.groupby(['importer','product']).ngroups}")

#### Cell 16 – Task 2: Prophet on All Series + Identify Underfitters

In [ ]:
print("\nRunning Prophet on Task 2 (Algeria exports)...")
prophet_fc_t2, prophet_metrics_t2, underfit_t2 = compute_prophet_metrics(
    df_t2, 'algeria_export_v', val_t2, test_t2, forecast_years=list(range(TRAIN_START, FORECAST_YEARS[-1]+1))
)

print(f"Prophet Task2 – {len(prophet_fc_t2)} pairs forecasted. Underfitters: {len(underfit_t2)}")

#### Cell 17 – Task 2: LSTM on Underfitting Series

In [ ]:
lstm_features_t2 = ['algeria_export_v', 'total_value', 'world_demand_growth', 
                    'gdp_growth_rate', 'inflation_rate', 'is_covid_year']
lstm_features_t2 = [c for c in lstm_features_t2 if c in df_t2.columns]

lstm_fc_t2 = {}
if underfit_t2:
    print(f"Running LSTM on {len(underfit_t2)} underfitting pairs for Task2...")
    for imp, prod in tqdm(underfit_t2):
        series = df_t2[(df_t2['importer']==imp) & (df_t2['product']==prod)].sort_values('year')
        if len(series) < SEQ_LEN + 3:
            continue
        fc = run_lstm_series(series, 'algeria_export_v', lstm_features_t2, FORECAST_YEARS)
        if fc:
            lstm_fc_t2[(imp, prod)] = fc
else:
    print("No underfitting pairs for Task2 (Prophet sufficient).")

#### Cell 18 – Task 2: Combine Final Forecasts

In [ ]:
final_fc_t2 = {}
for (imp, prod), fc_df in prophet_fc_t2.items():
    if (imp, prod) in underfit_t2 and (imp, prod) in lstm_fc_t2:
        for yr, val in lstm_fc_t2[(imp, prod)].items():
            fc_df.loc[fc_df['year'] == yr, 'yhat'] = val
            fc_df.loc[fc_df['year'] == yr, 'yhat_lower'] = val * 0.8
            fc_df.loc[fc_df['year'] == yr, 'yhat_upper'] = val * 1.2
    final_fc_t2[(imp, prod)] = fc_df

print(f"Task2 final forecasts prepared for {len(final_fc_t2)} pairs.")

#### Cell 19 – Task 2 Evaluation

In [ ]:
print("\n=== TASK 2 EVALUATION (Algeria Exports) ===")

val_df_t2 = pd.DataFrame(prophet_metrics_t2['val'])
if not val_df_t2.empty:
    print("\nProphet – Validation 2022 (all top-50 pairs):")
    print(f"  Avg MAE : {val_df_t2['MAE'].mean():14,.0f}")
    print(f"  Avg RMSE: {val_df_t2['RMSE'].mean():14,.0f}")
    print(f"  Avg MAPE: {val_df_t2['MAPE'].mean():10.1f}%")

test_df_t2 = pd.DataFrame(prophet_metrics_t2['test'])
if not test_df_t2.empty:
    print("\nProphet – Test 2023-2024 (all top-50 pairs):")
    print(f"  Avg MAE : {test_df_t2['MAE'].mean():14,.0f}")
    print(f"  Avg RMSE: {test_df_t2['RMSE'].mean():14,.0f}")
    print(f"  Avg MAPE: {test_df_t2['MAPE'].mean():10.1f}%")

#### Cell 20 – Load Opportunity Labels from Classification

In [ ]:
opp_df = pd.read_csv(CLASSIF_PATH) if os.path.exists(CLASSIF_PATH) else None
if opp_df is not None:
    opp_df = opp_df[['importer', 'product', 'opportunity_label']]
    print(f"Loaded {len(opp_df)} opportunity labels from classification.")
else:
    print("Classification results not found. Opportunity labels will be missing in dashboard.")

#### Cell 21 – Dashboard: Plot Sample Pairs

In [ ]:
def plot_dashboard_pair(imp, prod, fc_t1_dict, fc_t2_dict, hist_t1_df, hist_t2_df, opp_df):
    """Plot historical + forecast for a single (importer, product) pair."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # World demand
    hist1 = hist_t1_df[(hist_t1_df['importer']==imp) & (hist_t1_df['product']==prod)]
    fc1 = fc_t1_dict.get((imp, prod))
    ax1.plot(hist1['year'], hist1['total_value'], 'o-', color='#2563EB', label='Historical')
    if fc1 is not None:
        fc1_fut = fc1[fc1['year'].isin(FORECAST_YEARS)]
        ax1.plot(fc1_fut['year'], fc1_fut['yhat'].clip(0), 's--', color='#DC2626', label='Forecast')
        ax1.fill_between(fc1_fut['year'], fc1_fut['yhat_lower'].clip(0), fc1_fut['yhat_upper'].clip(0), alpha=0.2, color='#DC2626')
    ax1.axvspan(2020, 2021, alpha=0.1, color='orange')
    ax1.set_title(f'{imp} | {prod} – World Demand')
    ax1.legend()
    
    # Algeria exports
    hist2 = hist_t2_df[(hist_t2_df['importer']==imp) & (hist_t2_df['product']==prod)]
    fc2 = fc_t2_dict.get((imp, prod))
    ax2.plot(hist2['year'], hist2['algeria_export_v'], 'o-', color='#16A34A', label='Historical')
    if fc2 is not None:
        fc2_fut = fc2[fc2['year'].isin(FORECAST_YEARS)]
        ax2.plot(fc2_fut['year'], fc2_fut['yhat'].clip(0), 's--', color='#DC2626', label='Forecast')
        ax2.fill_between(fc2_fut['year'], fc2_fut['yhat_lower'].clip(0), fc2_fut['yhat_upper'].clip(0), alpha=0.2, color='#DC2626')
    ax2.axvspan(2020, 2021, alpha=0.1, color='orange')
    
    # Opportunity label
    opp_label = opp_df[(opp_df['importer']==imp) & (opp_df['product']==prod)]['opportunity_label'].values[0] if opp_df is not None else 'N/A'
    ax2.set_title(f'Algeria Exports – Opportunity: {opp_label}')
    ax2.legend()
    
    plt.tight_layout()
    return fig

# Prepare historical data
hist_t1 = filter_sparse_series(df, 'total_value', 1)[['importer','product','year','total_value']]
hist_t2 = filter_top_algeria_series(df, 50)[['importer','product','year','algeria_export_v']]

# Pick first 6 pairs from Task2 (or Task1) for dashboard
sample_pairs = list(final_fc_t2.keys())[:6] or list(final_fc_t1.keys())[:6]

for imp, prod in sample_pairs:
    fig = plot_dashboard_pair(imp, prod, final_fc_t1, final_fc_t2, hist_t1, hist_t2, opp_df)
    fig.savefig(os.path.join(DASH_DIR, f'dashboard_{imp}_{prod}.png'), dpi=100, bbox_inches='tight')
    plt.close(fig)

print(f"Dashboard plots saved to {DASH_DIR}")

#### Cell 22 – Save Final Forecasts to CSV (Optional)

In [ ]:
# Save Task2 forecasts (most relevant for strategy) as a clean table
forecast_rows = []
for (imp, prod), fc_df in final_fc_t2.items():
    for yr in FORECAST_YEARS:
        row = fc_df[fc_df['year'] == yr]
        if not row.empty:
            forecast_rows.append({
                'importer': imp,
                'product': prod,
                'year': yr,
                'forecast_algeria_export_v': row['yhat'].values[0],
                'lower_bound': row['yhat_lower'].values[0],
                'upper_bound': row['yhat_upper'].values[0]
            })
forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv(os.path.join(OUT_DIR, 'algeria_export_forecast_2025_2027.csv'), index=False)
print(f"Saved final forecasts to {OUT_DIR}/algeria_export_forecast_2025_2027.csv")

#### Cell 23 – Print Summary

In [ ]:
print("\n" + "="*50)
print("FORECASTING COMPLETED")
print("="*50)
print(f"Task1: {len(final_fc_t1)} pairs forecasted (world demand)")
print(f"Task2: {len(final_fc_t2)} pairs forecasted (Algeria exports)")
print(f"Underfitters replaced by LSTM: Task1={len(underfit_t1)}, Task2={len(underfit_t2)}")
print(f"Outputs saved in {OUT_DIR}")